## Dataset test, Kaggle predicciones

In [ ]:
import pandas as pd
import joblib


feature_columns = joblib.load("feature_columns.pkl")
robust_scaler = joblib.load("robust_scaler.pkl")
standard_scaler = joblib.load("standard_scaler.pkl")
modelo = joblib.load("mejor_modelo_red_neuronal.pkl")

NUM_COLS = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

# Test de Kaggle
df_testeo1 = pd.read_csv("test.csv")
df_testeo = df_testeo1.copy()

#Se rellenan los datos faltantes
for col in NUM_COLS:
    if col in df_testeo.columns:
        df_testeo[col] = df_testeo[col].fillna(df_testeo[col].median())

for col in ['HomePlanet', 'Destination', 'Cabin']:
    if col in df_testeo.columns:
        df_testeo[col] = df_testeo[col].fillna(df_testeo[col].mode()[0])

# Binarización
bool_cols = ['CryoSleep', 'VIP']
for col in bool_cols:
    if col in df_testeo.columns:
        df_testeo[col] = df_testeo[col].fillna(False).infer_objects(copy=False).astype(int)

# Extracción de los datos de cabin
if 'Cabin' in df_testeo.columns:
    df_testeo[['Deck', 'CabinNum', 'Side']] = df_testeo['Cabin'].str.split('/', expand=True)
    df_testeo['CabinNum'] = pd.to_numeric(df_testeo['CabinNum'], errors='coerce')
    df_testeo['CabinNum'] = df_testeo['CabinNum'].fillna(df_testeo['CabinNum'].median())
    df_testeo = df_testeo.drop(columns=['Cabin'], errors='ignore')

# Se quita el passanger Id
passenger_ids = df_testeo['PassengerId'].copy()
df_testeo = df_testeo.drop(columns=['PassengerId', 'Name'], errors='ignore')

# eNcoding
categorical_cols = ['HomePlanet', 'Destination', 'Deck', 'Side']
actual_categorical_cols = [col for col in categorical_cols if col in df_testeo.columns]

df_encoded = pd.get_dummies(
    df_testeo,
    columns=actual_categorical_cols,
    drop_first=True,
    dtype=int
)

# Escalar con Robust
for col in NUM_COLS:
    if col not in df_encoded.columns:
        df_encoded[col] = 0

df_encoded[NUM_COLS] = robust_scaler.transform(df_encoded[NUM_COLS])

# Los gastos escalados
columnas_gastos = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
df_encoded['Total_Spent'] = df_encoded[columnas_gastos].sum(axis=1)
df_encoded['Has_Expenses'] = (df_encoded['Total_Spent'] > 0).astype(int)

# sE HACE UN DF
df_final = df_encoded.reindex(columns=feature_columns, fill_value=0)

# Standar scaler
X_test_scaled = standard_scaler.transform(df_final)


# Verificar features
n_esperadas = getattr(modelo, "n_features_in_", None)
if n_esperadas is not None and n_esperadas != X_test_scaled.shape[1]:
    raise ValueError(
        f"Desfase de columnas: el modelo espera {n_esperadas} features, "
        f"pero se construyeron {X_test_scaled.shape[1]}."
    )

# Predicción
predicciones = modelo.predict(X_test_scaled)
predicciones = predicciones.astype(bool)

df_testeo1['Transported'] = predicciones
df_testeo1.to_csv("test_predicciones.csv", index=False)

# Se revisa
print("Número de predicciones:", len(predicciones))
print("Número de filas:", len(df_testeo1))
print("Shape entrada al modelo:", X_test_scaled.shape)
print("\nPredicciones.")
print(df_testeo1[['PassengerId', 'Transported']].head())

Número de predicciones: 4277
Número de filas: 4277
Shape entrada al modelo: (4277, 23)

Predicciones.
  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01        False


/tmp/ipykernel_735/3720623566.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_testeo[col] = df_testeo[col].fillna(False).infer_objects(copy=False).astype(int)
/tmp/ipykernel_735/3720623566.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_testeo[col] = df_testeo[col].fillna(False).infer_objects(copy=False).astype(int)
